# 03 - From a leaky regression to a pre-trip estimate

Purpose: show, with numbers, why the usual fare-regression exercise is
not a prediction problem, and what remains once the leakage is removed.

The demonstration is the point. 'R2 = 0.98' looks like success until you
notice the model is reading the meter.


In [ ]:
import sys, pathlib
ROOT = pathlib.Path.cwd().parent
sys.path.insert(0, str(ROOT / 'src'))

from taxi.config import load_settings
from taxi.contract import Contract
from taxi.core import duck
from taxi.core.months import Month

settings = load_settings(ROOT / 'configs' / 'settings.yml')
contract = Contract.load(settings.contract_path)
print('contract', contract.version, contract.fingerprint)


In [ ]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.model_selection import train_test_split

from taxi.marts.builder import partitions_in
from taxi.ml.features import load_frame
from taxi.ml.split import load_split

available = partitions_in(settings.paths.features)
print('feature months:', [str(m) for m in available])


## 1. The leaky model

Fit fare on realised distance and duration, evaluated on a random split.
Both inputs are the meter's own arguments.


In [ ]:
frame = load_frame(settings, available, sample_rows=150_000)
frame = frame.dropna(subset=['fare_amount', 'eval_trip_distance',
                             'eval_duration_minutes'])

leaky_X = frame[['eval_trip_distance', 'eval_duration_minutes']]
y = frame['fare_amount']
Xtr, Xte, ytr, yte = train_test_split(leaky_X, y, test_size=0.25, random_state=42)
leaky = LinearRegression().fit(Xtr, ytr)
pred = leaky.predict(Xte)
print(f'leaky model  MAE={mean_absolute_error(yte, pred):.3f}  R2={r2_score(yte, pred):.4f}')


That number is not a forecast. At the moment a quote is needed, neither
input exists. The model has been handed the answer in two pieces.

## 2. What is actually knowable before the trip

The contract already records this, so the notebook does not get to have
an opinion about it.


In [ ]:
audit = pd.DataFrame(contract.ml_fields())
audit[audit.ml_role.isin(['feature', 'conditional', 'target'])]


## 3. The same split discipline as the pipeline

Past to future. A random split lets December inform March.


In [ ]:
split = load_split(settings).restrict_to(available)
print(split.describe())

train = load_frame(settings, split.train, sample_rows=200_000)
test = load_frame(settings, split.test)
print(len(train), len(test))


## 4. Pre-trip models, temporally evaluated


In [ ]:
from taxi.ml.evaluate import compute_metrics
from taxi.ml.models import ODDistanceLookup, VARIANTS, build_model

results = []
for variant in VARIANTS.values():
    tr, te = train.copy(), test.copy()
    if variant.needs_od_lookup:
        lookup = ODDistanceLookup.fit(tr)      # training rows only
        tr['od_median_distance'] = lookup.transform(tr)
        te['od_median_distance'] = lookup.transform(te)
    for name in ['median_baseline', 'grouped_baseline', 'linear_regression',
                 'hist_gradient_boosting']:
        model = build_model(name, variant)
        model.fit(tr[variant.features], tr['fare_amount'])
        metrics = compute_metrics(te['fare_amount'].to_numpy(),
                                  model.predict(te[variant.features]))
        results.append({'variant': variant.name, 'model': name,
                        **metrics.as_dict()})
pd.DataFrame(results).round(4)


Read this table against the baselines, not against the leaky model. The
question is whether the pre-trip model beats a median lookup, and by how
much - not whether it reaches the leaky R2, which it never will and
should not.


## 5. Random split vs temporal split, same features

The gap between these two numbers is the size of the mistake a random
split would have hidden.


In [ ]:
variant = VARIANTS['destination_known']
combined = pd.concat([train, test], ignore_index=True)
lookup = ODDistanceLookup.fit(combined)
combined['od_median_distance'] = lookup.transform(combined)

Xtr, Xte, ytr, yte = train_test_split(combined[variant.features],
                                      combined['fare_amount'],
                                      test_size=0.25, random_state=42)
model = build_model('hist_gradient_boosting', variant).fit(Xtr, ytr)
random_metrics = compute_metrics(yte.to_numpy(), model.predict(Xte))
print('random split  ', random_metrics.line())
temporal = [r for r in results if r['variant'] == variant.name
            and r['model'] == 'hist_gradient_boosting'][0]
print('temporal split MAE=%.3f RMSE=%.3f R2=%.3f' %
      (temporal['mae'], temporal['rmse'], temporal['r2']))


## 6. Where the model fails


In [ ]:
from taxi.ml.evaluate import segment_errors

te = test.copy()
te['od_median_distance'] = ODDistanceLookup.fit(train).transform(te)
final = build_model('hist_gradient_boosting', variant)
tr = train.copy()
tr['od_median_distance'] = ODDistanceLookup.fit(train).transform(tr)
final.fit(tr[variant.features], tr['fare_amount'])
predictions = final.predict(te[variant.features])

segments = pd.DataFrame(segment_errors(te, te['fare_amount'].to_numpy(),
                                       predictions, settings,
                                       settings.ml['segments']))
segments.sort_values('mae', ascending=False).head(20)


In [ ]:
import matplotlib.pyplot as plt

for kind in ['distance_bucket', 'hour_of_day', 'fare_bucket']:
    view = segments[segments.segment_kind == kind].sort_values('segment_value')
    if view.empty:
        continue
    fig, ax = plt.subplots(figsize=(11, 3))
    ax.bar(view.segment_value, view.mae)
    ax.set_title(f'Test MAE by {kind}'); plt.xticks(rotation=45)
    plt.show()


## Conclusion to write here

- the leaky R2, and why it is not a result
- the pre-trip MAE against both baselines, per variant
- the random-vs-temporal gap
- the segments where the estimate is not usable, and what would fix them
  (routing distance, finer location, a rate-code signal)

Then run `taxi ml train && taxi report` so the numbers land in the
evidence file rather than only in this notebook.
